In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import sklearn.model_selection
import subprocess, tqdm, json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

In [ ]:
from aifes import keras2aifes

In [ ]:
GPU_ID = 0
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus[GPU_ID], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[GPU_ID], True)
    except RuntimeError as e:
        print(e)

In [ ]:
def imshow(img, title):
    plt.imshow(img, cmap = "binary")
    plt.title(str(title))
    plt.show()

def plot_value_array(i, predictions_array, true_label, num_classes):
    true_label = true_label[i]
    plt.grid(False)
    plt.xticks(range(num_classes))
    plt.yticks([])
    thisplot = plt.bar(range(num_classes), predictions_array, color="#777777")
    plt.ylim([0, 1])
    predicted_label = np.argmax(predictions_array)

    thisplot[predicted_label].set_color('red')
    thisplot[true_label].set_color('blue')

def plot_image(i, predictions_array, true_label, img):
    true_label, img = true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])

    plt.imshow(img, cmap=plt.cm.binary)

    predicted_label = np.argmax(predictions_array)
    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'

    plt.xlabel("{} {:2.0f}% ({})".format(predicted_label,
                                    100*np.max(predictions_array),
                                    true_label),
                                    color=color)

def plot_n_images(n, x, y):
    num_cols = 4
    assert (n % num_cols == 0)
    num_rows = n // num_cols
    plt.figure(figsize=(2*2*num_cols, 2*num_rows))
    for i in range(n):
        plt.grid(False)
        plt.xticks([])
        plt.yticks([])
        plt.subplot(num_rows, 2*num_cols, 2*i+2)
        plt.imshow(x[i])
        plt.title(str(y[i]))
        plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plt.tight_layout()
    plt.show()

def print_prediction(n, x_test, y_test, predictions, num_classes = 10):
    num_cols = 4
    assert (n % num_cols == 0)
    num_rows = n // num_cols
    plt.figure(figsize=(2*2*num_cols, 2*num_rows))
    for i in range(n):
        ind = np.random.randint(0, x_test.shape[0]-1)
        plt.subplot(num_rows, 2*num_cols, 2*i+1)
        plot_image(ind, predictions[ind], y_test, x_test)
        plt.subplot(num_rows, 2*num_cols, 2*i+2)
        plot_value_array(ind, predictions[ind], y_test, num_classes)
    plt.tight_layout()
    plt.show()

def plot_accuracy_loss(history, epochs, batch_size):
    stepX, stepY = 10, 0.025
    fig, axs = plt.subplots(2)
    axs[0].set_xticks(np.arange(0, epochs+stepX, stepX))
    axs[0].set_yticks(np.arange(0., 1.+stepY, stepY))
    axs[0].plot(history.epoch, history.history["accuracy"], 'g', label='Training accuracy')
    axs[0].plot(history.epoch, history.history["val_accuracy"], 'r', label='Validation accuracy')
    axs[0].set_title('Training & validation accuracy (batch_size = ' + str(batch_size) + ")")
    axs[0].set_xlabel('Epochs')
    axs[0].set_ylabel('Accuracy')

    axs[1].set_xticks(np.arange(0, epochs+stepX, stepX))
    axs[1].plot(history.epoch, history.history["loss"], 'g', label='Training loss')
    axs[1].plot(history.epoch, history.history["val_loss"], 'r', label='Validation loss')
    axs[1].set_title('Training & validation loss (batch_size = ' + str(batch_size) + ")")
    axs[1].set_xlabel('Epochs')
    axs[1].set_ylabel('Loss')
    fig.legend()
    plt.show()

def plot_dataset_balance(y, name):
    size           = y.shape[0]
    labels, counts = np.unique(y, return_counts = True)
    print("Minimum label occurence:", np.min(counts))
    plt.bar(labels, counts)
    plt.title(name + " set balance")
    plt.xlabel("Classes")
    plt.ylabel("Frequency")
    plt.show()


In [ ]:
def balance_dataset(x, y):
    number_classes = np.unique(y).shape[0]
    min_occurence  = np.min(np.unique(y, return_counts = True)[1])
    x_bal, y_bal   = [], []
    for c in tqdm.trange(number_classes): # For every class
        c_idx = np.where(y == c)[0] # Filter data per class
        x_bal.append(x[c_idx][:min_occurence]) # Only keep the first min_occurence images of each class
        y_bal.append(y[c_idx][:min_occurence])
    x_bal = np.concatenate(x_bal)
    y_bal = np.concatenate(y_bal)
    return x_bal, y_bal

def to_categorical(y):
    return np.eye(np.unique(y).shape[0])[y]

def hwc_to_chw(x):
    return np.transpose(x, (0, 3, 1, 2))

def chw_to_hwc(x):
    return np.transpose(x, (0, 2, 3, 1))

def normalize_dataset(x):
    return (x / 255) - 0.5

def flatten_dataset(x):
    return np.reshape(x, (x.shape[0], np.prod(x.shape[1:])))

# MNIST dataset loading/processing

In [ ]:
H, W, C     = 28, 28, 1
NUM_CLASSES = 10

# Building MNIST dataset
(x_mnist_train, y_mnist_train), (x_mnist_test, y_mnist_test) = tf.keras.datasets.mnist.load_data()

# Normalize dataset
x_mnist_train = normalize_dataset(x_mnist_train)
x_mnist_test = normalize_dataset(x_mnist_test)

print("Training set")
print(x_mnist_train.shape)
plot_dataset_balance(y_mnist_train, "Train")

print("Testing set")
print(x_mnist_test.shape)
plot_dataset_balance(y_mnist_test, "Test")

In [ ]:
print("Balanced training set")
x_mnist_train, y_mnist_train = balance_dataset(x_mnist_train, y_mnist_train)
print(x_mnist_train.shape)
plot_dataset_balance(y_mnist_train, "Balanced train")

print("Balaned testing set")
x_mnist_test, y_mnist_test = balance_dataset(x_mnist_test, y_mnist_test)
print(x_mnist_test.shape)
plot_dataset_balance(y_mnist_test, "Balanced test")

# Shuffle train set
p = np.random.permutation(y_mnist_train.shape[0])
x_mnist_train = x_mnist_train[p]
y_mnist_train = y_mnist_train[p]

# Shuffle test set
p = np.random.permutation(y_mnist_test.shape[0])
x_mnist_test = x_mnist_test[p]
y_mnist_test = y_mnist_test[p]

In [ ]:
# Print an image of each class
x = np.array([x_mnist_train[np.where(y_mnist_train == i)[0][0]] for i in range(NUM_CLASSES)] + [x_mnist_train[0]]*(NUM_CLASSES%4))
y = np.array([y_mnist_train[np.where(y_mnist_train == i)[0][0]] for i in range(NUM_CLASSES)] + [y_mnist_train[0]]*(NUM_CLASSES%4))
plot_n_images(x.shape[0], x, y)

# MNIST finetuning

In [ ]:
EPOCHS        = 20#50
LEARNING_RATE = 1e-3
BATCH_SIZE    = 128#256
LOSS_FUNCTION = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = False)
OPTIMIZER     = tf.keras.optimizers.Adam(LEARNING_RATE)

In [ ]:
model_mnist = tf.keras.models.Sequential([
    tf.keras.layers.Input((C*H*W,)),
    tf.keras.layers.Dense(512, activation = "relu"),
    tf.keras.layers.Dense(256, activation = "relu"),
    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(64, activation = "relu"),
    tf.keras.layers.Dense(10, activation = "softmax")
])
model_mnist.summary()

In [ ]:
OPTIMIZER     = tf.keras.optimizers.Adam(LEARNING_RATE)
model_mnist.compile(
    optimizer = OPTIMIZER,
    loss      = LOSS_FUNCTION,
    metrics   = ["accuracy"]
)

In [ ]:
x_mnist_train = flatten_dataset(x_mnist_train)
#y_mnist_train = to_categorical(y_mnist_train)

x_mnist_test = flatten_dataset(x_mnist_test)
#y_mnist_test = to_categorical(y_mnist_test)

history_mnist = model_mnist.fit(
    x_mnist_train,
    y_mnist_train,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_data  = (x_mnist_test, y_mnist_test)
)

plot_accuracy_loss(history_mnist, EPOCHS, BATCH_SIZE)

In [ ]:
print("Evaluation on train set : ")
model_mnist.evaluate(x_mnist_train, y_mnist_train, verbose=2)

print("Evaluation on test set : ")
model_mnist.evaluate(x_mnist_test, y_mnist_test, verbose=2)

In [ ]:
predictions = model_mnist.predict(x_mnist_test)
print_prediction(32, x_mnist_test.reshape((-1, H, W))+0.5, y_mnist_test, predictions, num_classes = NUM_CLASSES)

In [ ]:
cm = confusion_matrix(y_mnist_test, np.argmax(predictions, axis = 1))
disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = [str(i) for i in range(NUM_CLASSES)])
disp.plot(cmap = 'Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Save model and its metadata
model_name = "5dense-mnist-10class-finetuned_1"
model_mnist.save(model_name + ".h5")
with open(model_name + "-history.json", "w") as f:
    json.dump(history_mnist.history, f)

In [ ]:
model = tf.keras.models.load_model("5dense-mnist-10class-finetuned_1.h5")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("5dense-mnist-10class-finetuned.tflite", "wb") as f:
    f.write(tflite_model)

In [ ]:
# Reload model and its metadata
model_mnist_name = "5dense-mnist-10class-finetuned_1"
model_mnist = tf.keras.models.load_model(model_mnist_name + ".h5")
class History:
    def __init__(self):
        self.history = {}
        self.epoch   = 0
history_mnist = History()
with open(model_mnist_name + "-history.json", 'r') as f:
    history_mnist.history = json.load(f)
history_mnist.epoch       = list(range(len(history_mnist.history["loss"])))

# Model deployment using AIfES-Converter

In [28]:
help(keras2aifes.convert_to_fnn_f32)

Help on function convert_to_fnn_f32 in module aifes.keras2aifes:

convert_to_fnn_f32(keras_model: keras.src.engine.training.Model, output_path: str, with_weights=True)
    Converts the given Keras model to AIfES F32 (non-express version). Creates one header file with the model in the output_path that can be included to any AIfES project. If weights are extracted as well, a second header file is created, which contains the flattened weights of the Keras model.
    
    :param keras_model: Keras model, which should be converted
    :param output_path: File path, where the converted model should be stored. If the folder doesn't exist, it will be created by this function.
    :param with_weights: Extracts the weights and bias from the given model and creates a separate header file with flattened weights.



In [25]:
NUM_CLASSES = 10
NUM_PICTURES = 10
x_mnist_train = x_mnist_train[:NUM_PICTURES]
x_mnist_test = x_mnist_test[:NUM_PICTURES]
y_mnist_train = y_mnist_train[:NUM_PICTURES]
y_mnist_test = y_mnist_test[:NUM_PICTURES]

In [29]:
import inspect
from aifes import keras2aifes

print(inspect.signature(keras2aifes.convert_to_fnn_f32))

(keras_model: keras.src.engine.training.Model, output_path: str, with_weights=True)


In [33]:
import inspect
print(inspect.getsource(keras2aifes.convert_to_fnn_f32))

import importlib.metadata

print(importlib.metadata.version("AIfES-Converter"))

def convert_to_fnn_f32(keras_model: keras.Model, output_path: str, with_weights=True):
    """
    Converts the given Keras model to AIfES F32 (non-express version). Creates one header file with the model in the output_path that can be included to any AIfES project. If weights are extracted as well, a second header file is created, which contains the flattened weights of the Keras model.

    :param keras_model: Keras model, which should be converted
    :param output_path: File path, where the converted model should be stored. If the folder doesn't exist, it will be created by this function.
    :param with_weights: Extracts the weights and bias from the given model and creates a separate header file with flattened weights.

    """
    keras_extractor = KerasExtractor(keras_model, use_transposed_layers=False)
    aifestype = AifesType.NORMAL

    aifescode = AifesCodeGenerator(keras_extractor, 'default', aifestype, output_path, with_weights)
    aifescode.run_f32()

1.0.0


In [35]:
BASE_PATH     = "."
MODEL          = model_mnist
MODEL_TYPE     = "mlp"
DATASET_NAME   = "mnist"
LOSS_NAME      = "crossentropy"
OPTIMIZER_NAME = "adam"#"sgd"
FROZEN_LAYERS = [False, False, False, False, False]
WITH_WEIGHTS = True

CONVERT_DATASET = False

frozen_layers  = FROZEN_LAYERS
with_weights   = WITH_WEIGHTS
train_set_size = x_mnist_train.shape[0]
test_set_size  = x_mnist_test.shape[0]
test_name      = DATASET_NAME + "-" + MODEL_TYPE + "-" + str(len(MODEL.layers)) + "layers" + "-" + str(frozen_layers.count(True)) + "frozen" + "-" + ("pretrained" if with_weights else "untrained") + "-" + LOSS_NAME + "-" + OPTIMIZER_NAME

import os

test_name = (
    DATASET_NAME + "-" +
    MODEL_TYPE + "-" +
    str(len(MODEL.layers)) + "layers" + "-" +
    str(frozen_layers.count(True)) + "frozen" + "-" +
    ("pretrained" if with_weights else "untrained") + "-" +
    LOSS_NAME + "-" +
    OPTIMIZER_NAME
)

# Create output directory
f32_path = os.path.join(BASE_PATH, "f32")
os.makedirs(f32_path, exist_ok=True)

# F32 deployment
keras2aifes.convert_to_fnn_f32(
    MODEL,
    os.path.join(f32_path, test_name),
    with_weights
)
if(CONVERT_DATASET):
    dataset2flatbuffer.dataset2flatbuffer(
        BASE_PATH + "/dataset/" + DATASET_NAME + "-f32",
        32,
        x_mnist_train,
        y_mnist_train,
        x_mnist_test,
        y_mnist_test,
        num_classes = NUM_CLASSES
    )
